In [ ]:
# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a research paper (e.g., ai_research.pdf).
# - Chunking: The chatbot splits the paper into manageable sections so no detail is lost.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, making the paper searchable by meaning rather than keywords.
# - Retriever: When someone asks, “What does this paper say about reinforcement learning?”, the retriever pulls the most relevant chunks.
# - LLM Response: The Hugging Face model (Flan-T5) generates a concise, human-readable answer using those chunks as context.
# - Chat Loop: The team can keep asking questions interactively, like a research assistant that knows the paper inside out.

# ==========================================================
# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------
# chromadb → vector database
# sentence-transformers → embedding model
# pypdf → reading PDF files
# transformers → running the LLM

!pip install chromadb sentence-transformers pypdf transformers


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os

# Library for reading PDF documents
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# HuggingFace model pipeline
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Load the PDF Document
# ----------------------------------------------------------
# This document acts as the knowledge source for RAG

print("Loading PDF document...")

reader = PdfReader("/content/ai_research.pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(text))

# Preview some text
print("\nPreview:\n")
print(text[:500])


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------
# LLMs work better with smaller text segments
# so we split the document into chunks

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------
# Convert each chunk into a numerical vector
# These vectors allow semantic similarity search

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------
# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------
# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------
# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------
# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------
# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_rounds=max_rounds)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:


# ==========================================================
# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------
# chromadb → vector database
# sentence-transformers → embedding model
# pypdf → reading PDF files
# transformers → running the LLM

!pip install chromadb sentence-transformers pypdf transformers


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os

# Library for reading PDF documents
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# HuggingFace model pipeline
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Load the PDF Document
# ----------------------------------------------------------
# This document acts as the knowledge source for RAG

print("Loading PDF document...")

reader = PdfReader("/content/data_privacy_regulation.pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(text))

# Preview some text
print("\nPreview:\n")
print(text[:500])


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------
# LLMs work better with smaller text segments
# so we split the document into chunks

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------
# Convert each chunk into a numerical vector
# These vectors allow semantic similarity search

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------
# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------
# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------
# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------
# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------
# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Loading PDF document...
Document Loaded
Total Characters: 2885

Preview:

Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.
Article 1: Definitions

Personal Data: Any information relating to an identified or identifiable individual.

Data Contr

Splitting document into chunks...
Total Chunks Created: 7

Example Chunk:

Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
p

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Old collection deleted
New vector collection created

Creating embeddings and storing in ChromaDB...
All chunks stored successfully

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully

RAG Chatbot Ready
Type 'exit' to stop

Ask a question: Explain RAG


Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
ta.

Individuals have the right to request deletion of personal data under certain conditions.

Individuals may object to automated decision■making processes involving their data.
Article 4: Cross■Border Data Transfers
Transfer of personal data to another country or jurisdiction is permitted only when the receiving
country ensures an adequate level of data protection. Organizations must implement safeguards
such as standard contractual clauses, binding corporate rules, or regulatory approvals  Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers,

KeyboardInterrupt: Interrupted by user

In [ ]:


# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED


# STEP 0 — Install Required Libraries

# chromadb → vector database
# sentence-transformers → embedding model
# pypdf → reading PDF files
# transformers → running the LLM

# !pip install chromadb sentence-transformers pypdf transformers



# STEP 1 — Import Libraries


import os

# Library for reading PDF documents
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# HuggingFace model pipeline
from transformers import pipeline


# STEP 2 — Load the PDF Document

# This document acts as the knowledge source for RAG

print("Loading PDF document...")

reader = PdfReader("/content/Introduction_to_Data_Science.pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(text))

# Preview some text
print("\nPreview:\n")
print(text[:500])


# STEP 3 — Chunk the Document

# LLMs work better with smaller text segments
# so we split the document into chunks

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])


# STEP 4 — Create Embeddings

# Convert each chunk into a numerical vector
# These vectors allow semantic similarity search

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# STEP 5 — Create Vector Database

# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# STEP 6 — Store Chunks in Vector DB


print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# STEP 7 — Retriever Function

# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# STEP 8 — Load the LLM

# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# STEP 9 — Question Answering Function

# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]



# STEP 10 — Chat Loop

# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Loading PDF document...
Document Loaded
Total Characters: 3427

Preview:

Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data scientists analyze this data to help businesses make better decisions.
1. What is Data Science?
Data Science involves collecting, cleaning, analyzing, 

Splitting document into chunks...
Total Chunks Created: 8

Example Chunk:

Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Old collection deleted
New vector collection created

Creating embeddings and storing in ChromaDB...
All chunks stored successfully

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully

RAG Chatbot Ready
Type 'exit' to stop

Ask a question: Tools Used in Data Science


Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
Science involves collecting, cleaning, analyzing, and interpreting large datasets. It uses
techniques from statistics, machine learning, and data visualization to identify patterns and trends
within data.
Key Components of Data Science:

Data Collection – Gathering raw data from different sources.

Data Cleaning – Removing noise, errors, or missing values.

Data Analysis – Applying statistical and computational methods.

Machine Learning – Building models that learn patterns from data.

Dat Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms.

KeyboardInterrupt: Interrupted by user

In [ ]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    result = response[0]["generated_text"]

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b19bb2621eb03d9f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Scenario: "The Healthcare Policy Navigator"
# Background
# You are part of the Healthcare Compliance & Policy Team at a large hospital network. New government regulations on patient data privacy and telemedicine practices have just been released. The hospital must quickly adapt to ensure compliance and avoid penalties.
# Challenge
# The hospital uploads a PDF of the healthcare regulation into the Policy Navigator (your Gradio + Chroma + LLM app). Your task is to:
# - Process the regulation document so the assistant can store and understand it.
# - Ask compliance-related questions about patient rights, telemedicine rules, and penalties.
# - Generate clear, actionable answers that can guide doctors, administrators, and IT staff



# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

# !pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a HealthCare AI assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    result = response[0]["generated_text"]

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜  HealthCare AI RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0c54ed497f8aa5273b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Scenario: "The Environmental Policy Compliance Assistant"
# Background
# You are part of the Sustainability & Environmental Compliance Team at a global manufacturing company.
# New government regulations on carbon emissions, waste disposal, and renewable energy adoption have just been released.
# The company must ensure compliance to avoid fines and reputational damage.

# Challenge
# The company uploads a PDF of the environmental regulation into the Compliance Assistant (Gradio + Chroma + LLM app).
# Your task is to:
# - Process the regulation document so the assistant can store and understand it.
# - Ask compliance-related questions about emission limits, waste management rules, and renewable energy targets.
# - Generate clear, actionable answers that can guide engineers, sustainability officers, and executives.

# Roles
# - Learner (You): Environmental compliance officer using the assistant.
# - Assistant (The RAG App): Provides answers strictly based on uploaded environmental regulations.
# - Stakeholders: Plant managers, sustainability officers, and executives who need concise compliance guidance.

# 🔄 Flow of the Scenario
# - Upload Environmental Regulation PDF
# Example: “National Carbon Emissions Act 2026”.
# - System Processes Document
# - Splits into chunks.
# - Embeds into vector database.
# - Stores for retrieval.
# - Ask Questions
# - “What is the maximum carbon emission allowed per factory per year?”
# - “What penalties apply if hazardous waste is not disposed of properly?”
# - “What renewable energy targets must we meet by 2030?”
# - Assistant Responds
# - Retrieves relevant chunks.
# - Generates compliance-focused answers.
# - Provides short, clear guidance.
# - Outcome
# - Learners practice extracting environmental obligations.
# - Managers receive summarized compliance insights.
# - Executives gain confidence in sustainability strategy alignment.

# 🎯 Training Objective
# This scenario helps learners:
# - Understand how RAG systems can support environmental compliance.
# - Practice formulating precise queries to extract obligations.
# - Experience how AI can simplify complex environmental regulations into actionable steps.

# 👉 Would you like me to also draft a sample regulation PDF text (like the healthcare one I created earlier)
#  for this environmental context, so you can upload it into your assistant and simulate queries?


# Scenario: "The Healthcare Policy Navigator"
# Background
# You are part of the Healthcare Compliance & Policy Team at a large hospital network. New government regulations on patient data privacy and telemedicine practices have just been released. The hospital must quickly adapt to ensure compliance and avoid penalties.
# Challenge
# The hospital uploads a PDF of the healthcare regulation into the Policy Navigator (your Gradio + Chroma + LLM app). Your task is to:
# - Process the regulation document so the assistant can store and understand it.
# - Ask compliance-related questions about patient rights, telemedicine rules, and penalties.
# - Generate clear, actionable answers that can guide doctors, administrators, and IT staff



# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

# !pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a HealthCare AI assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    result = response[0]["generated_text"]

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("The Environmental Policy Compliance Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()



Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1afdb4d6d6c545ead0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
